# Part 2: 다양한 모델 포맷 - 실습

- 실습 1: PyTorch 모델 저장 방식 이해
- 실습 2: ONNX 변환과 검증
- 실습 3: ONNX Runtime으로 추론하기

In [ ]:
# 필요한 라이브러리 설치
!pip install onnx onnxruntime onnxscript -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.5/17.5 MB 52.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 693.4/693.4 kB 29.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 6.5 MB/s eta 0:00:00


In [2]:
import torch
import torch.nn as nn
import numpy as np
import os

print(f"PyTorch 버전: {torch.__version__}")

PyTorch 버전: 2.9.0+cu126


---
## 실습 1: PyTorch 모델 저장 방식 이해

### 학습 목표
- state_dict vs 전체 모델 저장의 차이 이해
- 모델 저장/로드 방법 익히기
- 저장된 파일 크기 비교

### 1-1. 간단한 CNN 모델 정의

In [3]:
class SimpleCNN(nn.Module):
    """MNIST 분류를 위한 간단한 CNN 모델"""
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))  # 28x28 -> 14x14
        x = self.pool(self.relu(self.conv2(x)))  # 14x14 -> 7x7
        x = x.view(-1, 64 * 7 * 7)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# 모델 생성
model = SimpleCNN()
print(model)

# 파라미터 수 확인
total_params = sum(p.numel() for p in model.parameters())
print(f"\n총 파라미터 수: {total_params:,}개")

SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=3136, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)

총 파라미터 수: 421,642개


### 1-2. 모델 저장 방법 비교
---

✅ 1) **state_dict 저장 (추천)**

  - 가중치(weight)만 저장
  - 모델 구조 코드는 따로 필요
  - 장점: 유연성, 용량 작음, 호환성 좋음
  - 단점: 모델 클래스 정의 필요

```
torch.save(model.state_dict(), "model.pth")

model = MyModel()
model.load_state_dict(torch.load("model.pth"))
model.eval()
```


⚠️ 2) 전체 모델 저장 (비추천)

  - 모델 구조 + 가중치 통째로 저장
  - 로드시 저장할 때의 디렉토리 구조나 클래스 정의가 불러올 때와 완벽하게 일치해야 함.
  - 장점: 로드가 간단
  - 단점: 코드/클래스 경로 바뀌면 로드 실패 가능
```
torch.save(model, "full_model.pth")

model = torch.load("full_model.pth")
model.eval()
```


In [4]:
# 저장 디렉토리 생성
os.makedirs('saved_models', exist_ok=True)

# 방법 1: state_dict만 저장 (권장)
torch.save(model.state_dict(), 'saved_models/model_state_dict.pth')

# 방법 2: 전체 모델 저장 (pickle 사용)
torch.save(model, 'saved_models/model_full.pth')

# 파일 크기 비교
size_state_dict = os.path.getsize('saved_models/model_state_dict.pth')
size_full = os.path.getsize('saved_models/model_full.pth')

print("모델 저장 방식 비교")
print("=" * 50)
print(f"state_dict 저장: {size_state_dict:,} bytes ({size_state_dict/1024:.2f} KB)")
print(f"전체 모델 저장:  {size_full:,} bytes ({size_full/1024:.2f} KB)")

모델 저장 방식 비교
state_dict 저장: 1,690,251 bytes (1650.64 KB)
전체 모델 저장:  1,692,151 bytes (1652.49 KB)


### 1-3. 저장된 모델 불러오기

In [10]:
# 방법 1: state_dict 불러오기
model_loaded = SimpleCNN()    # 모델 구조를 먼저 정의
state_dict = torch.load(
    "saved_models/model_state_dict.pth",
    map_location="cpu",
    weights_only=True,
)
model_loaded.load_state_dict(state_dict)
model_loaded.eval()           # 평가 모드로 설정
print("state_dict 로드 완료!")


# 방법 2: 전체 모델 불러오기
model_full_loaded = torch.load(
    "saved_models/model_full.pth",
    map_location="cpu",
    weights_only=False,
)
model_full_loaded.eval()
print("전체 모델 로드 완료!")

# 두 모델이 같은 결과를 출력하는지 확인
dummy_input = torch.randn(1, 1, 28, 28)
with torch.no_grad():
    output1 = model_loaded(dummy_input)
    output2 = model_full_loaded(dummy_input)

print(f"\n---- 두 모델의 출력이 동일한가? ----: {torch.allclose(output1, output2)}")

state_dict 로드 완료!
전체 모델 로드 완료!

---- 두 모델의 출력이 동일한가? ----: True


### 🎯 실습 1-1: state_dict 내용 확인하기

state_dict에 저장된 가중치의 구조와 shape을 확인해보기
1) state_dict
    - 모델이 학습한 파라미터(가중치/편향)들의 딕셔너리
    - key는 레이어 이름, value는 텐서(weight/bias)
    - 예: fc1.weight, fc1.bias

2) key, value
    - key : 파라미터 이름 (어떤 레이어인지)
    - value : 그 파라미터 값(텐서)
    - value.shape : 파라미터 행렬의 크기 (예: [128, 784])
    - dtype : 저장 형식 (보통 float32)
3) numel()
    - value.numel() = 텐서 안에 숫자가 총 몇 개 들어있는지
    - 이걸 다 합친 것이 total_elements

In [12]:
state_dict = model.state_dict()

print("모델이 저장하는 값(state_dict) 확인하기")
print("=" * 70)

total_elements = 0
for key, value in state_dict.items():
    num_elements = value.numel()
    total_elements += num_elements
    print(f"{key:25} | shape: {str(value.shape):20} | dtype: {value.dtype} | elements: {num_elements:,}")

print("=" * 70)
print(f"총 파라미터 개수(가중치 숫자 개수): {total_elements:,}")
print(f"FP32로 저장했을 때 필요한 메모리(용량): {total_elements * 4 / 1024:.2f} KB")

state_dict 내용 분석
conv1.weight              | shape: torch.Size([32, 1, 3, 3]) | dtype: torch.float32 | elements: 288
conv1.bias                | shape: torch.Size([32])     | dtype: torch.float32 | elements: 32
conv2.weight              | shape: torch.Size([64, 32, 3, 3]) | dtype: torch.float32 | elements: 18,432
conv2.bias                | shape: torch.Size([64])     | dtype: torch.float32 | elements: 64
fc1.weight                | shape: torch.Size([128, 3136]) | dtype: torch.float32 | elements: 401,408
fc1.bias                  | shape: torch.Size([128])    | dtype: torch.float32 | elements: 128
fc2.weight                | shape: torch.Size([10, 128]) | dtype: torch.float32 | elements: 1,280
fc2.bias                  | shape: torch.Size([10])     | dtype: torch.float32 | elements: 10
총 파라미터 개수(가중치 숫자 개수): 421,642
FP32로 저장했을 때 필요한 메모리(용량): 1647.04 KB


---
## 실습 2: ONNX 변환과 검증

### 학습 목표
- PyTorch 모델을 ONNX로 변환하는 방법 익히기
- ONNX 모델 구조 확인
- 변환 전후 결과 검증

### 2-1. PyTorch → ONNX 변환

✅ **ONNX란?**

ONNX는 PyTorch 모델을 다른 환경(ONNX Runtime, TensorRT 등)에서도 실행할 수 있게 변환하는 표준 포맷이에요.

✅ **torch.onnx.export() 주요 파라미터 설명**
- model : 변환할 PyTorch 모델
- dummy_input : 모델 입력 예시(입력 shape 결정)
- onnx_path : 저장할 파일 경로
- export_params=True : 학습된 가중치(weight) 포함
- opset_version=18 : ONNX 연산 규칙 버전(표준 버전)
- do_constant_folding=True : 고정 계산은 미리 계산해 최적화
- input_names / output_names : 입력/출력 이름 지정
- dynamic_axes : batch size를 가변으로 설정

In [15]:
import onnx

# 모델을 평가 모드로 설정 (!!!!!!!!!중요!!!!!!!!)
model.eval()

# 더미 입력 생성 (MNIST: 1채널, 28x28)
dummy_input = torch.randn(1, 1, 28, 28)

# ONNX로 내보내기
onnx_path = 'saved_models/model.onnx'

torch.onnx.export(
    model,                    # 변환할 모델
    dummy_input,              # 모델 입력 예시
    onnx_path,                # 저장 경로
    export_params=True,       # 모델 파라미터 포함
    opset_version=18,         # ONNX opset 버전: ONNX 연산 규칙(표준 버전)을 18로 맞춤
    do_constant_folding=True, # 상수 폴딩 최적화
    input_names=['input'],    # 입력 이름
    output_names=['output'],  # 출력 이름
    dynamic_axes={            # 동적 배치 크기 지원
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"ONNX 모델 저장 완료: {onnx_path}")
print(f"파일 크기: {os.path.getsize(onnx_path) / 1024:.2f} KB")

/tmp/ipython-input-2339855264.py:12: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `SimpleCNN([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decomposition...
[torch.onnx] Run decomposition... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
ONNX 모델 저장 완료: saved_models/model.onnx
파일 크기: 11.75 KB


### 2-2. ONNX 모델 검증

In [16]:
# ONNX 모델 로드 및 검증
onnx_model = onnx.load(onnx_path)

# 모델 유효성 검사
try:
    onnx.checker.check_model(onnx_model)
    print("✅ ONNX 모델 검증 성공!")
except Exception as e:
    print(f"❌ 모델 검증 실패: {e}")

✅ ONNX 모델 검증 성공!


### 2-3. ONNX 모델 구조 확인

In [17]:
# 모델 메타데이터
print("ONNX 모델 정보")
print("=" * 50)
print(f"IR 버전: {onnx_model.ir_version}")
print(f"Producer: {onnx_model.producer_name}")
print(f"Model 버전: {onnx_model.model_version}")

# 입력 정보
print("\n입력 정보:")
for input in onnx_model.graph.input:
    print(f"  이름: {input.name}")
    tensor_type = input.type.tensor_type
    dims = [d.dim_value if d.dim_value else d.dim_param for d in tensor_type.shape.dim]
    print(f"  Shape: {dims}")

# 출력 정보
print("\n출력 정보:")
for output in onnx_model.graph.output:
    print(f"  이름: {output.name}")

ONNX 모델 정보
IR 버전: 10
Producer: pytorch
Model 버전: 0

입력 정보:
  이름: input
  Shape: ['s77', 1, 28, 28]

출력 정보:
  이름: output


### 2-4. 연산자(Op) 목록 확인

In [18]:
# 모델에 사용된 연산자 목록
op_types = set()
for node in onnx_model.graph.node:
    op_types.add(node.op_type)

print("사용된 ONNX 연산자 목록:")
print("=" * 30)
for op in sorted(op_types):
    print(f"  - {op}")

사용된 ONNX 연산자 목록:
  - Conv
  - Gemm
  - MaxPool
  - Relu
  - Reshape


### 🎯 실습 2-1: PyTorch와 ONNX 출력 비교

같은 입력에 대해 PyTorch 모델과 ONNX 모델의 출력이 동일한지 확인해보기

In [21]:
import numpy as np
import torch
import onnxruntime as ort

# ============================================================
# ✅ 목표: PyTorch 모델 출력과 ONNX Runtime 출력이 "거의 같은지" 확인하기
#
# 왜 "완전히 같아야" 하지 않고 "거의 같은지" 보나?
# - FP32/FP16 연산 순서, 최적화(상수 폴딩), 런타임 구현 차이 때문에
#   아주 작은 오차(반올림 오차)가 생길 수 있음
# - 그래서 tolerance(허용 오차)를 두고 비교하는 것이 일반적
# ============================================================

# 1) 테스트 입력 생성 (모델이 받는 입력 형태와 동일해야 함: [batch, channel, H, W])
test_input = torch.randn(1, 1, 28, 28)

# 2) PyTorch 모델 추론
model.eval()  # ✅ Dropout/BatchNorm 같은 레이어가 있으면 eval 모드가 중요
with torch.no_grad():  # ✅ 추론에서는 gradient 계산이 필요 없어서 메모리/속도 절약
    pytorch_output = model(test_input).cpu().numpy()  # numpy로 변환 (비교용)

# 3) ONNX Runtime으로 추론
# - ort.InferenceSession은 ONNX 모델을 로드해서 실행하는 객체
# - 입력은 numpy 배열로 넣어야 함
ort_session = ort.InferenceSession(onnx_path)

onnx_output = ort_session.run(
    None,  # None이면 "모든 출력"을 반환
    {"input": test_input.cpu().numpy()}  # ✅ export할 때 input_names=['input'] 했기 때문에 키가 'input'
)[0]  # 출력이 여러 개면 리스트로 오므로 첫 번째 출력만 사용

# 4) 출력 일부 확인 (상위 5개 값만 프린트)
print("PyTorch 출력 (상위 5개):")
print(pytorch_output[0][:5])

print("\nONNX 출력 (상위 5개):")
print(onnx_output[0][:5])

# 5) 오차 계산
# max_diff = 두 출력 사이에서 가장 큰 절대 차이 (최악의 케이스)
max_diff = np.abs(pytorch_output - onnx_output).max()
print(f"\n최대 차이(max abs diff): {max_diff:.2e}")

# ------------------------------------------------------------
# ✅ 동일 여부(allclose)란?
# np.allclose(A, B, atol=..., rtol=...) 는
# "A와 B가 허용 오차 범위 안에서 거의 같은가?"를 True/False로 알려줌
#
# - atol(absolute tolerance): 절대 오차 허용치 (여기서는 1e-5)
#   예: |A - B| <= 1e-5 이면 '같다'고 인정
#
# - rtol(relative tolerance): 상대 오차 허용치 (기본값도 존재)
#   값의 크기가 클 때는 상대 오차 기준도 함께 고려됨
#
# ✅ 왜 tolerance가 필요?
# - ONNX 변환/런타임 최적화 때문에 아주 미세한 반올림 차이가 생길 수 있음
# - 그래서 "완벽히 동일" 대신 "실용적으로 동일"을 판단함
# ------------------------------------------------------------
is_same = np.allclose(pytorch_output, onnx_output, atol=1e-5)
print(f"동일 여부 (허용 오차 atol=1e-5): {is_same}")


PyTorch 출력 (상위 5개):
[-0.20871101  0.10812928 -0.19496955  0.05748934  0.04447991]

ONNX 출력 (상위 5개):
[-0.20871104  0.10812925 -0.19496953  0.05748926  0.04447993]

최대 차이(max abs diff): 7.82e-08
동일 여부 (허용 오차 atol=1e-5): True


---
## 실습 3: ONNX Runtime으로 추론하기

### 학습
- ONNX Runtime 사용법 익히기
- 추론 속도 비교
- 배치 처리 실습

### 3-1. ONNX Runtime 세션 생성

In [22]:
import onnxruntime as ort

# 사용 가능한 프로바이더 확인
print("사용 가능한 Execution Providers:")
print(ort.get_available_providers())

# CPU 기반 세션 생성
session_options = ort.SessionOptions()
session_options.graph_optimization_level = ort.GraphOptimizationLevel.ORT_ENABLE_ALL

ort_session = ort.InferenceSession(
    onnx_path,
    sess_options=session_options,
    providers=['CPUExecutionProvider']
)

print("\nONNX Runtime 세션 생성 완료!")

사용 가능한 Execution Providers:
['AzureExecutionProvider', 'CPUExecutionProvider']

ONNX Runtime 세션 생성 완료!


### 3-2. 입출력 정보 확인

In [24]:
# 입력 정보
print("입력 정보:")
for input_meta in ort_session.get_inputs():
    print(f"  이름: {input_meta.name}")
    print(f"  Shape: {input_meta.shape}")
    print(f"  Type: {input_meta.type}")

print()

# 출력 정보
print("출력 정보:")
for output_meta in ort_session.get_outputs():
    print(f"  이름: {output_meta.name}")
    print(f"  Shape: {output_meta.shape}")
    print(f"  Type: {output_meta.type}")

입력 정보:
  이름: input
  Shape: ['s77', 1, 28, 28]
  Type: tensor(float)

출력 정보:
  이름: output
  Shape: ['s77', 10]
  Type: tensor(float)


### 3-3. 추론 속도 비교: PyTorch vs ONNX Runtime

In [25]:
import time

# 테스트 설정
batch_size = 32
num_iterations = 100

# 테스트 데이터
test_input_torch = torch.randn(batch_size, 1, 28, 28)
test_input_numpy = test_input_torch.numpy()

# PyTorch 추론 시간 측정
model.eval()
with torch.no_grad():
    # Warm-up
    for _ in range(10):
        _ = model(test_input_torch)

    # 측정
    start = time.time()
    for _ in range(num_iterations):
        _ = model(test_input_torch)
    pytorch_time = (time.time() - start) / num_iterations * 1000  # ms

# ONNX Runtime 추론 시간 측정
# Warm-up
for _ in range(10):
    _ = ort_session.run(None, {'input': test_input_numpy})

# 측정
start = time.time()
for _ in range(num_iterations):
    _ = ort_session.run(None, {'input': test_input_numpy})
onnx_time = (time.time() - start) / num_iterations * 1000  # ms

print(f"추론 속도 비교 (배치 크기: {batch_size}, 반복: {num_iterations}회)")
print("=" * 50)
print(f"PyTorch:      {pytorch_time:.3f} ms/batch")
print(f"ONNX Runtime: {onnx_time:.3f} ms/batch")
print(f"\n속도 향상: {pytorch_time/onnx_time:.2f}x")

추론 속도 비교 (배치 크기: 32, 반복: 100회)
PyTorch:      12.287 ms/batch
ONNX Runtime: 3.550 ms/batch

속도 향상: 3.46x


### 🎯 실습 3-1: 배치 크기별 처리량 비교

다양한 배치 크기에서 ONNX Runtime의 처리량(throughput)을 측정해보기

In [26]:
batch_sizes = [1, 4, 16, 32, 64, 128]
num_iterations = 50

print("배치 크기별 처리량 (ONNX Runtime)")
print("=" * 60)
print(f"{'Batch Size':^12} | {'Latency (ms)':^15} | {'Throughput (samples/s)':^20}")
print("-" * 60)

for batch_size in batch_sizes:
    test_input = np.random.randn(batch_size, 1, 28, 28).astype(np.float32)

    # Warm-up
    for _ in range(5):
        _ = ort_session.run(None, {'input': test_input})

    # 측정
    start = time.time()
    for _ in range(num_iterations):
        _ = ort_session.run(None, {'input': test_input})
    elapsed = time.time() - start

    latency_ms = (elapsed / num_iterations) * 1000
    throughput = (batch_size * num_iterations) / elapsed

    print(f"{batch_size:^12} | {latency_ms:^15.3f} | {throughput:^20.1f}")

배치 크기별 처리량 (ONNX Runtime)
 Batch Size  |  Latency (ms)   | Throughput (samples/s)
------------------------------------------------------------
     1       |      0.209      |        4790.8       
     4       |      0.462      |        8666.0       
     16      |      1.704      |        9391.7       
     32      |      3.749      |        8534.7       
     64      |      6.828      |        9373.8       
    128      |     14.494      |        8831.5       


### 3-4. 파일 크기 비교

In [27]:
# 모든 저장된 파일 크기 비교
files = {
    'PyTorch state_dict (.pth)': 'saved_models/model_state_dict.pth',
    'PyTorch 전체 모델 (.pth)': 'saved_models/model_full.pth',
    'ONNX 모델 (.onnx)': 'saved_models/model.onnx',
}

print("모델 파일 크기 비교")
print("=" * 60)

for name, path in files.items():
    if os.path.exists(path):
        size = os.path.getsize(path)
        print(f"{name:35} | {size/1024:>10.2f} KB")

모델 파일 크기 비교
PyTorch state_dict (.pth)           |    1650.64 KB
PyTorch 전체 모델 (.pth)                |    1652.49 KB
ONNX 모델 (.onnx)                     |      11.75 KB


---
## 📝 Part 2 정리

### 핵심 개념
1. **PyTorch 저장 방식**
   - `state_dict`: 가중치만 저장 (권장)
   - 전체 모델 저장: 코드 의존성 있음

2. **ONNX (Open Neural Network Exchange)**
   - 프레임워크 독립적인 표준 포맷
   - 다양한 추론 엔진에서 사용 가능
   - `torch.onnx.export()`로 변환

3. **ONNX Runtime**
   - 고성능 크로스플랫폼 추론 엔진
   - CPU, GPU, NPU 등 다양한 하드웨어 지원
   - 자동 그래프 최적화

### 변환 파이프라인
```
PyTorch 모델 → ONNX → ONNX Runtime (추론)
                  ↘ TensorRT (NVIDIA GPU 최적화)
                  ↘ CoreML (Apple 디바이스)
                  ↘ OpenVINO (Intel 최적화)
```
